# Module 5: End-to-End ML Pipeline

**Snowpark ML Fundamentals - Week 1 | Lunch & Learn**

---

## Learning Objectives
- Build a complete ML pipeline with `snowflake.ml.modeling.pipeline.Pipeline`
- Chain preprocessing + model in a single object
- Register the pipeline in the Model Registry
- Understand Snowpark vs Stored Procedures for production

## Key Concept
> **Snowpark ML Pipeline** mirrors scikit-learn's `Pipeline` - chain preprocessing
> and model into a single fit/predict object. Deploy to the Model Registry
> for production inference.

---

## 5.1 Setup

In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

from snowpark_fundamentals.session import create_session
from snowpark_fundamentals.data.sample_data import create_customer_churn_dataset
from snowpark_fundamentals.data.loader import split_data

session = create_session(env_path='../.env')

NUMERIC_COLS = ['AGE', 'TENURE_MONTHS', 'MONTHLY_CHARGES', 'TOTAL_CHARGES',
                'SUPPORT_TICKETS', 'USAGE_HOURS_PER_WEEK', 'NUM_DEPENDENTS']
CATEGORICAL_COLS = ['PLAN_TYPE', 'CONTRACT_TYPE', 'PAYMENT_METHOD']
LABEL_COL = 'CHURNED'

df = create_customer_churn_dataset(session, n_rows=5000)
train_df, test_df = split_data(df)
print(f"Train: {train_df.count()} | Test: {test_df.count()}")

## 5.2 Build the Pipeline

A single `Pipeline` object that handles:
1. StandardScaler for numeric features
2. OrdinalEncoder for categorical features
3. XGBClassifier for prediction

In [ ]:
from snowpark_fundamentals.modeling.pipeline import build_ml_pipeline, fit_and_predict

pipeline = build_ml_pipeline(
    numeric_cols=NUMERIC_COLS,
    categorical_cols=CATEGORICAL_COLS,
    label_col=LABEL_COL,
    model_params={
        'n_estimators': 100,
        'max_depth': 6,
        'learning_rate': 0.1,
    },
)

print("Pipeline steps:")
for name, step in pipeline.steps:
    print(f"  {name}: {type(step).__name__}")

In [ ]:
# Fit and predict in one go
fitted_pipeline, predictions = fit_and_predict(pipeline, train_df, test_df)

predictions.select(LABEL_COL, 'PREDICTION').show(10)

## 5.3 Evaluate the Pipeline

In [ ]:
from snowpark_fundamentals.modeling.evaluation import evaluate_binary_classifier

metrics = evaluate_binary_classifier(predictions, LABEL_COL, 'PREDICTION')
print("Pipeline Performance:")
for metric, value in metrics.items():
    print(f"  {metric:>12}: {value:.4f}")

## 5.4 Register the Pipeline

The entire pipeline (preprocessing + model) is registered as a single deployable unit.

In [ ]:
from snowpark_fundamentals.registry.model_registry import get_registry, log_model

current_db = session.get_current_database().replace('"', '')
current_schema = session.get_current_schema().replace('"', '')
registry = get_registry(session, current_db, current_schema)

# Register the full pipeline
all_input_cols = NUMERIC_COLS + CATEGORICAL_COLS
model_version = log_model(
    registry=registry,
    model=fitted_pipeline,
    model_name='CHURN_PIPELINE',
    version_name='V1',
    sample_input=test_df.select(all_input_cols).limit(10),
    metrics=metrics,
)

print("Full pipeline registered as CHURN_PIPELINE v1")
print(f"Metrics: {metrics}")

## 5.5 Snowpark vs Stored Procedures

### When to use each:

| Scenario | Recommendation |
|----------|---------------|
| **Interactive development** | Snowpark client (notebooks, IDE) |
| **EDA & prototyping** | Snowpark client |
| **Scheduled batch scoring** | Wrap in Stored Procedure + Snowflake Task |
| **CI/CD deployment** | Deploy as Stored Procedure via GitHub Actions |
| **Ad-hoc inference** | Model Registry (call from SQL or Python) |

### The recommended pattern:
1. **Develop** with Snowpark client (what we did today)
2. **Register** models in the Model Registry
3. **Operationalize** by wrapping in Stored Procedures for scheduling

### Example Stored Procedure (for production):
```python
# This would be deployed as a Stored Procedure
def score_customers(session):
    from snowflake.ml.registry import Registry
    registry = Registry(session)
    model = registry.get_model('CHURN_PIPELINE').version('V1')
    new_data = session.table('NEW_CUSTOMERS')
    predictions = model.run(new_data, function_name='predict')
    predictions.write.save_as_table('CHURN_PREDICTIONS', mode='overwrite')
    return f"Scored {predictions.count()} customers"
```

## 5.6 Architecture Recap

```
Data in Snowflake
       |
       v
  Snowpark DataFrame  (lazy evaluation, distributed)
       |
       v
  Preprocessing        (StandardScaler, OrdinalEncoder - runs in warehouse)
       |
       v
  Model Training        (XGBoost, RandomForest - runs in warehouse)
       |
       v
  Model Registry        (versioned, governed, SQL-callable)
       |
       v
  Inference              (batch via SP/Tasks, or ad-hoc via Registry)
```

## Session Summary: What We Covered Today

| Module | Topic | Key API |
|--------|-------|---------|
| 1 | DataFrames | `session.table()`, `F.col()`, `.filter()`, `.group_by()` |
| 2 | Preprocessing | `StandardScaler`, `OneHotEncoder`, `OrdinalEncoder` |
| 3 | Model Training | `XGBClassifier`, `RandomForestClassifier`, `accuracy_score` |
| 4 | Model Registry | `Registry`, `log_model()`, `model.run()` |
| 5 | Pipeline | `Pipeline`, end-to-end flow, Snowpark vs SP |

### Coming Up Next Week: **Snowflake Model Registry** (Deep Dive)
- Advanced versioning strategies
- Model lifecycle management
- Deploying to Snowpark Container Services

In [ ]:
# Cleanup (optional)
# from snowpark_fundamentals.registry.model_registry import delete_model
# delete_model(registry, 'CHURN_PIPELINE')
# delete_model(registry, 'CHURN_PREDICTOR')
# session.sql('DROP TABLE IF EXISTS CUSTOMER_CHURN_DATA').collect()

session.close()
print("Session closed. Thank you!")